<a href="https://colab.research.google.com/github/OdinakaDico/Oasis3_dementia_classification_project/blob/main/Oasis3_dementia_group_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(torch.cuda.is_available())

Mounted at /content/drive
True


In [2]:
data_dir = "/content/drive/MyDrive/Colab Notebooks/processed_oasis_axis1_train_val_clean"

In [3]:
import os

print(os.listdir("/content/drive/MyDrive/Colab Notebooks/processed_oasis_axis1_train_val_clean"))

['val', 'train']


In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(
    root=data_dir + "/train",
    transform=train_transforms
)

val_dataset = datasets.ImageFolder(
    root=data_dir + "/val",
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(train_dataset.classes)
print(len(train_dataset), len(val_dataset))

['dementia', 'normal']
16000 4000


In [5]:
import os
import pandas as pd

records = []

for split in ["train", "val"]:
    for label in ["normal", "dementia"]:

        label_path = os.path.join(data_dir, split, label)

        for subject in os.listdir(label_path):

            subject_path = os.path.join(label_path, subject)

            png_count = len([
                f for f in os.listdir(subject_path)
                if f.endswith(".png")
            ])

            records.append({
                "split": split,
                "label": label,
                "subject": subject,
                "num_slices": png_count
            })

slice_count_df = pd.DataFrame(records)
problem_subjects = slice_count_df[slice_count_df["num_slices"] != 100]

print(problem_subjects)

print(slice_count_df.head())

print("\nTotal subjects:", len(slice_count_df))
print("Total slices:", slice_count_df["num_slices"].sum())

Empty DataFrame
Columns: [split, label, subject, num_slices]
Index: []
   split   label   subject  num_slices
0  train  normal  OAS31107         100
1  train  normal  OAS31110         100
2  train  normal  OAS31131         100
3  train  normal  OAS31133         100
4  train  normal  OAS31020         100

Total subjects: 200
Total slices: 20000


In [6]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# load pretrained ResNet18 (NEW WAY)
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# replace final layer (VERY IMPORTANT)
model.fc = nn.Linear(model.fc.in_features, 2)

# move to GPU
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [8]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss:.4f}")

Epoch [1/5], Loss: 205.4045
Epoch [2/5], Loss: 95.9275
Epoch [3/5], Loss: 58.2241
Epoch [4/5], Loss: 37.5798
Epoch [5/5], Loss: 29.7553


In [9]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 72.53%


In [10]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=val_dataset.classes))

[[1512  488]
 [ 611 1389]]
              precision    recall  f1-score   support

    dementia       0.71      0.76      0.73      2000
      normal       0.74      0.69      0.72      2000

    accuracy                           0.73      4000
   macro avg       0.73      0.73      0.72      4000
weighted avg       0.73      0.73      0.72      4000



In [13]:
torch.save(model.state_dict(), "resnet18_axis1_baseline_73.pth")